# Week 13: Production Hardening

**Lab companion to [week_13.md](../agenda/week_13.md)**

In this lab, you will:
1. Implement retry logic
2. Handle rate limits
3. Build fallback systems
4. Create production-ready clients

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import time
import random

load_dotenv()
client = OpenAI()

print("✓ Ready!")

## 1. Retry with Exponential Backoff

In [ ]:
import functools
from typing import Callable, TypeVar

T = TypeVar('T')

def retry_with_backoff(
    max_retries: int = 3,
    base_delay: float = 1.0,
    max_delay: float = 60.0,
    exponential_base: float = 2.0,
    jitter: bool = True
):
    """Decorator for retry with exponential backoff."""

    def decorator(func: Callable[..., T]) -> Callable[..., T]:
        @functools.wraps(func)
        def wrapper(*args, **kwargs) -> T:
            last_exception = None

            for attempt in range(max_retries + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    last_exception = e

                    if attempt == max_retries:
                        break

                    # Calculate delay
                    delay = min(base_delay * (exponential_base ** attempt), max_delay)

                    if jitter:
                        delay = delay * (0.5 + random.random())

                    print(f"  Retry {attempt + 1}/{max_retries} after {delay:.1f}s: {e}")
                    time.sleep(delay)

            raise last_exception

        return wrapper
    return decorator

# Example usage
@retry_with_backoff(max_retries=3)
def unreliable_function():
    if random.random() < 0.7:  # 70% failure rate
        raise Exception("Random failure")
    return "Success!"

try:
    result = unreliable_function()
    print(result)
except Exception as e:
    print(f"Final failure: {e}")

In [ ]:
# Apply to LLM calls
@retry_with_backoff(max_retries=3, base_delay=1.0)
def reliable_completion(messages: list, **kwargs) -> str:
    response = client.chat.completions.create(
        model=kwargs.get("model", "gpt-4o-mini"),
        messages=messages,
        **{k: v for k, v in kwargs.items() if k != "model"}
    )
    return response.choices[0].message.content

# Test
result = reliable_completion([{"role": "user", "content": "Say hello!"}])
print(result)

## 2. Rate Limiter

In [ ]:
import threading
from collections import deque

class TokenBucketRateLimiter:
    """Rate limiter using token bucket algorithm."""

    def __init__(self, rate: float, capacity: int):
        """
        Args:
            rate: Tokens added per second
            capacity: Maximum tokens in bucket
        """
        self.rate = rate
        self.capacity = capacity
        self.tokens = capacity
        self.last_update = time.time()
        self.lock = threading.Lock()

    def acquire(self, tokens: int = 1) -> float:
        """Acquire tokens, returning wait time if needed."""
        with self.lock:
            now = time.time()

            # Add tokens based on time elapsed
            elapsed = now - self.last_update
            self.tokens = min(self.capacity, self.tokens + elapsed * self.rate)
            self.last_update = now

            if self.tokens >= tokens:
                self.tokens -= tokens
                return 0.0

            # Calculate wait time
            wait_time = (tokens - self.tokens) / self.rate
            return wait_time

    def wait_and_acquire(self, tokens: int = 1):
        """Wait if necessary and acquire tokens."""
        wait_time = self.acquire(tokens)
        if wait_time > 0:
            print(f"  Rate limited, waiting {wait_time:.1f}s...")
            time.sleep(wait_time)
            self.acquire(tokens)  # Acquire after waiting

# Test
limiter = TokenBucketRateLimiter(rate=2, capacity=5)  # 2 requests/sec, burst of 5

print("Making 8 requests with rate limit:")
for i in range(8):
    limiter.wait_and_acquire()
    print(f"  Request {i + 1} at {time.time() % 100:.1f}")

## 3. Fallback Provider

In [ ]:
class FallbackLLM:
    """LLM client with fallback models."""

    def __init__(self):
        self.client = OpenAI()
        self.models = [
            "gpt-4o-mini",  # Primary
            "gpt-4o",       # Fallback 1
            "gpt-3.5-turbo" # Fallback 2
        ]

    def chat(self, messages: list, **kwargs) -> dict:
        """Try models in order until one succeeds."""

        last_error = None

        for model in self.models:
            try:
                print(f"  Trying: {model}")
                response = self.client.chat.completions.create(
                    model=model,
                    messages=messages,
                    **kwargs
                )
                return {
                    "content": response.choices[0].message.content,
                    "model": model,
                    "tokens": response.usage.total_tokens
                }
            except Exception as e:
                print(f"  ✗ {model} failed: {e}")
                last_error = e

        raise Exception(f"All models failed. Last error: {last_error}")

# Test
fallback = FallbackLLM()
result = fallback.chat([{"role": "user", "content": "Hi!"}])
print(f"\nResponse from {result['model']}: {result['content']}")

## 4. Circuit Breaker

In [ ]:
from enum import Enum

class CircuitState(Enum):
    CLOSED = "closed"    # Normal operation
    OPEN = "open"        # Failing, reject requests
    HALF_OPEN = "half_open"  # Testing if recovered

class CircuitBreaker:
    """Circuit breaker pattern implementation."""

    def __init__(
        self,
        failure_threshold: int = 5,
        recovery_timeout: float = 30.0,
        half_open_max_calls: int = 1
    ):
        self.failure_threshold = failure_threshold
        self.recovery_timeout = recovery_timeout
        self.half_open_max_calls = half_open_max_calls

        self.state = CircuitState.CLOSED
        self.failure_count = 0
        self.last_failure_time = None
        self.half_open_calls = 0

    def can_execute(self) -> bool:
        """Check if request can proceed."""
        if self.state == CircuitState.CLOSED:
            return True

        if self.state == CircuitState.OPEN:
            # Check if recovery timeout has passed
            if time.time() - self.last_failure_time >= self.recovery_timeout:
                self.state = CircuitState.HALF_OPEN
                self.half_open_calls = 0
                return True
            return False

        # HALF_OPEN - allow limited calls
        return self.half_open_calls < self.half_open_max_calls

    def record_success(self):
        """Record successful call."""
        if self.state == CircuitState.HALF_OPEN:
            self.state = CircuitState.CLOSED
        self.failure_count = 0

    def record_failure(self):
        """Record failed call."""
        self.failure_count += 1
        self.last_failure_time = time.time()

        if self.state == CircuitState.HALF_OPEN:
            self.state = CircuitState.OPEN
        elif self.failure_count >= self.failure_threshold:
            self.state = CircuitState.OPEN
            print(f"  ⚡ Circuit OPENED after {self.failure_count} failures")

# Test
breaker = CircuitBreaker(failure_threshold=3, recovery_timeout=5)

def call_with_breaker():
    if not breaker.can_execute():
        print("  Circuit is OPEN, request rejected")
        return None

    try:
        # Simulate 70% failure rate
        if random.random() < 0.7:
            raise Exception("Service error")
        breaker.record_success()
        return "Success"
    except Exception as e:
        breaker.record_failure()
        return f"Failed: {e}"

print("Testing circuit breaker:")
for i in range(10):
    result = call_with_breaker()
    print(f"  Call {i+1}: {result} (state: {breaker.state.value})")
    time.sleep(0.5)

## 5. Production-Ready Client

In [ ]:
class ProductionLLMClient:
    """Production-ready LLM client with all safeguards."""

    def __init__(
        self,
        rate_limit: float = 10.0,  # requests/sec
        max_retries: int = 3,
        timeout: float = 30.0
    ):
        self.client = OpenAI(timeout=timeout)
        self.rate_limiter = TokenBucketRateLimiter(rate_limit, int(rate_limit * 2))
        self.circuit_breaker = CircuitBreaker()
        self.max_retries = max_retries

        # Stats
        self.stats = {
            "requests": 0,
            "successes": 0,
            "failures": 0,
            "retries": 0,
            "rate_limited": 0
        }

    def chat(self, messages: list, **kwargs) -> dict:
        """Make a production-safe LLM call."""

        self.stats["requests"] += 1

        # Check circuit breaker
        if not self.circuit_breaker.can_execute():
            raise Exception("Circuit breaker is OPEN")

        # Rate limiting
        wait_time = self.rate_limiter.acquire()
        if wait_time > 0:
            self.stats["rate_limited"] += 1
            time.sleep(wait_time)

        # Retry loop
        last_error = None
        for attempt in range(self.max_retries + 1):
            try:
                response = self.client.chat.completions.create(
                    model=kwargs.get("model", "gpt-4o-mini"),
                    messages=messages,
                    **{k: v for k, v in kwargs.items() if k != "model"}
                )

                self.circuit_breaker.record_success()
                self.stats["successes"] += 1

                return {
                    "content": response.choices[0].message.content,
                    "model": response.model,
                    "tokens": response.usage.total_tokens,
                    "attempt": attempt + 1
                }

            except Exception as e:
                last_error = e

                if attempt < self.max_retries:
                    self.stats["retries"] += 1
                    delay = (2 ** attempt) * (0.5 + random.random())
                    time.sleep(delay)

        # All retries failed
        self.circuit_breaker.record_failure()
        self.stats["failures"] += 1
        raise last_error

    def get_stats(self) -> dict:
        """Get client statistics."""
        return {
            **self.stats,
            "success_rate": self.stats["successes"] / max(self.stats["requests"], 1),
            "circuit_state": self.circuit_breaker.state.value
        }

# Test
prod_client = ProductionLLMClient()

for i in range(5):
    try:
        result = prod_client.chat([{"role": "user", "content": f"Say {i+1}"}])
        print(f"Call {i+1}: OK (attempt {result['attempt']})")
    except Exception as e:
        print(f"Call {i+1}: Failed - {e}")

print(f"\nStats: {prod_client.get_stats()}")

## Summary

You learned:
- ✅ Retry with exponential backoff
- ✅ Token bucket rate limiting
- ✅ Fallback providers
- ✅ Circuit breaker pattern
- ✅ Production-ready LLM clients

**Next:** [Week 15 - Deployment & Fine-tuning](week_14_deployment.ipynb)